# 处理股票数据

In [1]:
!python scripts/dump_bin.py dump_all --data_path ".\.qlib\csv_stock_data\csv_data_上证50" --qlib_dir ".\.qlib\qlib_data\上证50_data" --symbol_field_name code --date_field_name datetime --freq day --include_fields open,high,low,close,volume,PB

2025-09-11 10:59:44.052 | INFO     | __main__:_get_all_date:307 - start get all date......

100%|██████████| 204/204 [00:03<00:00, 51.74it/s] 
2025-09-11 10:59:47.996 | INFO     | __main__:_get_all_date:326 - end of get all date.

2025-09-11 10:59:47.997 | INFO     | __main__:_dump_calendars:329 - start dump calendars......
2025-09-11 10:59:48.046 | INFO     | __main__:_dump_calendars:332 - end of calendars dump.

2025-09-11 10:59:48.046 | INFO     | __main__:_dump_instruments:335 - start dump instruments......
2025-09-11 10:59:48.048 | INFO     | __main__:_dump_instruments:337 - end of instruments dump.

2025-09-11 10:59:48.048 | INFO     | __main__:_dump_features:340 - start dump features......

100%|██████████| 204/204 [00:03<00:00, 54.53it/s]
2025-09-11 10:59:51.789 | INFO     | __main__:_dump_features:347 - end of features dump.



# 处理股票时间范围数据

In [3]:
from numpy import NAN
import pandas as pd

index_name="上证50_成分股时间数据.csv"

df=pd.read_csv(f".\.qlib\csv_time_data\{index_name}")
print(df.info())
print(df.head())
# 修复：使用isna()来正确检查NaN值
df_miss=df[df['S_CON_OUTDATE'].isna()]
print(f"缺失值数量: {len(df_miss)}")
print(df_miss.head())

# 查找出现不只一次的股票代码
print("\n=== 查找重复的股票代码 ===")
# 统计每个股票代码出现的次数
code_counts = df['CODE'].value_counts()
print(f"总共有 {len(df)} 条记录，涉及 {len(code_counts)} 个不同的股票代码")

# 找出出现不只一次的股票代码
duplicate_codes = code_counts[code_counts > 1]

# 显示重复股票代码的详细信息
if len(duplicate_codes) > 0:
    print(f"\n重复股票代码的详细信息:")
    for code in duplicate_codes.index:
        print(f"\n股票代码: {code} (出现 {duplicate_codes[code]} 次)")
        duplicate_records = df[df['CODE'] == code]
        print(duplicate_records)





<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   CODE           275 non-null    object
 1   S_CON_INDATE   275 non-null    object
 2   S_CON_OUTDATE  225 non-null    object
dtypes: object(3)
memory usage: 6.6+ KB
None
        CODE S_CON_INDATE S_CON_OUTDATE
0  601111.SH   2019-06-17    2020-06-12
1  601989.SH   2011-07-01    2020-12-11
2  601901.SH   2016-12-12    2017-12-08
3  601288.SH   2010-07-29           NaN
4  601066.SH   2019-06-17    2023-12-08
缺失值数量: 50
         CODE S_CON_INDATE S_CON_OUTDATE
3   601288.SH   2010-07-29           NaN
38  601398.SH   2006-11-10           NaN
41  601166.SH   2007-02-26           NaN
42  601088.SH   2007-10-23           NaN
43  601668.SH   2010-01-04           NaN

=== 查找重复的股票代码 ===
总共有 275 条记录，涉及 205 个不同的股票代码

重复股票代码的详细信息:

股票代码: 601669.SH (出现 4 次)
          CODE S_CON_INDATE S_CON_OUTDATE
159  601669

In [4]:
# 转换为all.txt格式
print("\n=== 转换为all.txt格式 ===")
# 处理缺失的S_CON_OUTDATE，用默认的结束日期填充，这里注意后续可能要更换为当前日期数据
df_converted = df.copy()
df_converted['S_CON_OUTDATE'] = df_converted['S_CON_OUTDATE'].fillna('2025-09-09')

# 选择需要的列并重命名
df_all_format = df_converted[['CODE', 'S_CON_INDATE', 'S_CON_OUTDATE']].copy()
df_all_format.columns = ['CODE', 'START_DATE', 'END_DATE']

# 按股票代码排序
df_all_format = df_all_format.sort_values('CODE')

print("转换后的数据格式:")
print(df_all_format.head(10))

# 保存为txt文件（制表符分隔）
output_file = ".qlib/qlib_data/上证50_data/instruments/上证50_all.txt"
df_all_format.to_csv(output_file, sep='\t', index=False, header=False)
print(f"\n数据已保存到: {output_file}")
print(f"总共 {len(df_all_format)} 条记录")

# 显示文件内容预览
print("\n文件内容预览:")
with open(output_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 10:  # 显示前10行
            print(line.strip())
        else:
            break


=== 转换为all.txt格式 ===
转换后的数据格式:
          CODE  START_DATE    END_DATE
47   600000.SH  2004-01-02  2022-06-10
256  600001.SH  2006-07-03  2009-06-30
252  600002.SH  2004-07-01  2006-04-21
242  600004.SH  2004-01-02  2007-01-22
12   600005.SH  2004-07-01  2011-06-30
145  600006.SH  2004-01-02  2005-06-30
241  600008.SH  2004-01-02  2005-06-30
80   600009.SH  2019-12-16  2021-12-10
231  600009.SH  2004-01-02  2009-12-31
121  600010.SH  2012-01-04  2016-12-09

数据已保存到: .qlib/qlib_data/上证50_data/instruments/上证50_all.txt
总共 275 条记录

文件内容预览:
600000.SH	2004-01-02	2022-06-10
600001.SH	2006-07-03	2009-06-30
600002.SH	2004-07-01	2006-04-21
600004.SH	2004-01-02	2007-01-22
600005.SH	2004-07-01	2011-06-30
600006.SH	2004-01-02	2005-06-30
600008.SH	2004-01-02	2005-06-30
600009.SH	2019-12-16	2021-12-10
600009.SH	2004-01-02	2009-12-31
600010.SH	2012-01-04	2016-12-09
